# 第 9 週：變遷偵測與驗證 — ARIA v6.0

## 課程資訊

**課程名稱**：遙測與空間資訊之分析與應用  
**學校**：國立臺灣大學 National Taiwan University (NTU)  
**授課教師**：蘇文瑄教授 Prof. Su Wen-Ray  
**案例主題**：馬太鞍堰塞湖（Matai'an Barrier Lake）災害監測與驗證

---

## 本週實驗主軸

本週作業聚焦於「災害事件的遙測定量分析」，目標不是只把圖做出來，而是要建立一套**可驗證、可解釋、可支援防災決策**的流程。  
我們將使用 Sentinel-2 多時期影像，觀察馬太鞍地區從災前、災中到災後的地表光譜變化，並透過地面驗證點來評估模型表現。

本次分析核心包含：

- 利用 STAC 自動搜尋三期 Sentinel-2 L2A 影像
- 使用 SCL（Scene Classification Layer）進行雲與雲影遮罩
- 計算 NDVI 與 NDWI，建立災前後差異圖
- 透過驗證點建立混淆矩陣與精度指標
- 以 F1-score 協助選擇閾值，建立三級信心分布圖
- 產出可支援防災判斷的 ARIA v6.0 驗證報告

---

## 實驗時間配置

| 實驗 | 主題 | 預估時間 | 說明 |
|------|------|----------|------|
| **Lab 1** | 差異圖製作（NDVI / NDWI） | 35 分鐘 | 觀察光譜變化與災害範圍 |
| **Lab 2** | 精度評估與驗證 | 50 分鐘 | 量化檢核模型可信度 |

---

## 背景脈絡

本案例以三個時間點建立災害敘事：

- **災前（Pre）**：2025-06-01，代表事件發生前的基準狀態
- **災中（Mid）**：2025-08-15，代表堰塞湖形成與最明顯的階段
- **災後（Post）**：2025-10-10，代表湖水消退或地表恢復階段

**研究區域 BBOX**：`[121.28, 23.56, 121.52, 23.76]`  
**主要指標**：

- **NDVI**：觀察植生狀態與植被變化
- **NDWI**：觀察水體範圍與積水分布

---

## 為什麼這份作業重要？

在大地工程防災與災害應變中，「有圖」不等於「有用」，更不等於「可信」。  
如果沒有正確處理雲遮罩、閾值與驗證程序，遙測影像很容易把雲影誤判成水體，或在高度不平衡的樣本下得到看似很高、但其實沒有決策價值的 Overall Accuracy。

因此，本週的重點是學會：

1. 如何把衛星影像轉換成有意義的災害資訊
2. 如何用定量指標證明分析結果是否可信
3. 如何把分析結果轉化成防災決策可理解的資訊


## Lab 1：差異圖製作 — 哪一個指標最能揭露堰塞湖？

### 實驗目標

本實驗將針對三期 Sentinel-2 影像計算光譜指標，並透過差異圖（Difference Mapping）觀察地表從災前到災中、再到災後的變化情形。  
我們希望回答的核心問題是：

- 堰塞湖的形成在哪一個光譜指標上最清楚？
- 哪些區域可能是水體增加、植生減少或崩塌後裸露地擴張？
- 如果改變閾值，水體範圍會如何變化？

---

### 主要工作流程

1. 從 STAC 搜尋研究區域的 Sentinel-2 L2A 三期影像
2. 建立並套用 SCL 雲遮罩，排除無效像素
3. 計算每一期的 NDVI 與 NDWI
4. 計算兩組時間差異：
   - `ΔIndex = Mid - Pre`
   - `ΔIndex = Post - Mid`
5. 以 2×2 圖表呈現不同指標與時期變化
6. 掃描多組 NDWI 閾值，觀察偵測面積如何改變

---

### 實驗重點提醒

- **差異圖不是分類圖**：它呈現的是「變化程度」，不是直接的地物類別
- **雲遮罩必須先做**：如果先算差異再處理雲，容易出現大量假變化
- **NDWI 通常比 NDVI 更適合水體**：但 NDVI 仍可協助理解植生與地表改變

---

### 預期成果

完成 Lab 1 後，你應該能夠：

- 判讀災前、災中、災後的主要水體變化區
- 比較 NDVI 與 NDWI 在本案例中的優缺點
- 理解閾值不是固定答案，而是和防災需求有關的決策


In [ ]:
# [S1] Environment Setup
# ──────────────────────────────────────────────────────────────────

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import seaborn as sns
from datetime import datetime
import warnings

from pystac_client import Client
import planetary_computer as pc
import stackstac
from pyproj import Transformer
from sklearn.metrics import confusion_matrix, f1_score

warnings.filterwarnings("ignore")

# Configuration
MATAIAN_BBOX = [121.28, 23.56, 121.52, 23.76]  # [W, S, E, N]

# ── Lake-focused AOI (NEW in W9) ──
# The barrier lake is ~1 km² in the upper-left of the 24×20 km scene.
# Using the full scene inflates OA with trivially-correct "stable" pixels.
# ── 堰塞湖只有 ~1 km²，用全場景會讓大量穩定像素灌水 OA。
LAKE_BBOX_LONLAT = [121.27, 23.68, 121.32, 23.72]  # ~5×4 km around lake

# ── SCL cloud masking (CRITICAL — without this, ΔNDWI shows "phantom water") ──
# SCL (Scene Classification Layer) classifies each pixel:
#   0=No data, 1=Saturated, 2=Dark, 3=Cloud shadow,
#   4=Vegetation, 5=Bare soil, 6=Water, 7=Unclassified,
#   8=Cloud (medium), 9=Cloud (high), 10=Thin cirrus, 11=Snow/ice
# We keep classes that represent clear-sky surface pixels.
# ── 保留代表晴空地表的類別。
SCL_CLEAR_CLASSES = [2, 4, 5, 6, 7, 11]

STAC_ENDPOINT = os.getenv("STAC_ENDPOINT", "https://planetarycomputer.microsoft.com/api/stac/v1")
S2_COLLECTION = "sentinel-2-l2a"
TARGET_EPSG = 32651
TARGET_RESOLUTION = 10
STREAM_CHUNKSIZE = 1024
SEARCH_WINDOWS_DAYS = [3, 10, 20, 45]

DEFAULT_NDWI_THRESHOLD = 0.10
CONFIDENCE_LOW = 0.05
CONFIDENCE_HIGH = 0.15
PIXEL_AREA_KM2 = (TARGET_RESOLUTION ** 2) / 1_000_000


def _stack_scene(item, assets, bbox=MATAIAN_BBOX):
    """Load one or more Sentinel-2 assets onto a common 10 m UTM grid."""
    stacked = stackstac.stack(
        [pc.sign(item)],
        assets=assets,
        epsg=TARGET_EPSG,
        resolution=TARGET_RESOLUTION,
        bounds_latlon=bbox,
        chunksize=STREAM_CHUNKSIZE,
    ).squeeze("time")
    return stacked.compute()


def _extract_grid_meta(stacked):
    """Extract raster coordinate metadata for lon/lat -> row/col conversion."""
    x = np.asarray(stacked["x"].values, dtype="float64")
    y = np.asarray(stacked["y"].values, dtype="float64")
    epsg = int(stacked.attrs.get("epsg", TARGET_EPSG))
    return {
        "x": x,
        "y": y,
        "dx": float(x[1] - x[0]),
        "dy": float(y[1] - y[0]),
        "epsg": epsg,
        "crs": f"EPSG:{epsg}",
        "transformer": Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True),
    }


def bbox_to_pixel_window(bbox_lonlat, grid):
    """Convert a lon/lat bounding box into row/col slices on the raster grid."""
    west, south, east, north = bbox_lonlat
    x0, y0 = grid["transformer"].transform(west, north)
    x1, y1 = grid["transformer"].transform(east, south)

    col0 = int(np.floor((x0 - grid["x"][0]) / grid["dx"]))
    col1 = int(np.ceil((x1 - grid["x"][0]) / grid["dx"]))
    row0 = int(np.floor((y0 - grid["y"][0]) / grid["dy"]))
    row1 = int(np.ceil((y1 - grid["y"][0]) / grid["dy"]))

    row_min, row_max = sorted((row0, row1))
    col_min, col_max = sorted((col0, col1))

    row_min = max(0, row_min)
    row_max = min(len(grid["y"]), row_max)
    col_min = max(0, col_min)
    col_max = min(len(grid["x"]), col_max)
    return slice(row_min, row_max), slice(col_min, col_max)


def stream_scl(item, bbox=MATAIAN_BBOX):
    """Stream SCL and keep only clear-sky pixels required by the lab."""
    scl = _stack_scene(item, assets=["SCL"], bbox=bbox)
    scl_values = scl.values.squeeze().astype("int16")
    return np.isin(scl_values, SCL_CLEAR_CLASSES)


OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Plot style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("✓ Environment setup complete")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Study area BBOX: {MATAIAN_BBOX}")
print(f"  STAC endpoint: {STAC_ENDPOINT}")


In [ ]:
# [S2] Search and Load Three-Act Scenes
# ──────────────────────────────────────────────────────────────────
# Search STAC for Pre, Mid, Post scenes of Matai'an barrier lake

# Scene dates (three acts of the disaster narrative)
SCENE_DATES = {
    "Pre": "2025-06-01",   # Baseline (before typhoon)
    "Mid": "2025-08-15",   # Lake present at peak
    "Post": "2025-10-10",  # Lake drained / recovered
}


def to_utc_timestamp(value):
    """Normalize datetime-like input to a UTC-aware pandas Timestamp."""
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        return ts.tz_localize("UTC")
    return ts.tz_convert("UTC")


def cloud_priority(cloud_cover):
    """Rank cloud cover into decision buckets so clear scenes beat cloudy scenes."""
    if pd.isna(cloud_cover):
        return 3
    if cloud_cover < 20:
        return 0
    if cloud_cover < 50:
        return 1
    if cloud_cover < 80:
        return 2
    return 3


def robust_search(client, bbox, target_date, collection=S2_COLLECTION):
    """
    Search Sentinel-2 L2A imagery around a target date.

    Strategy:
    1. Search within the widest teaching window around the target date.
    2. Prefer clear scenes first, then temporal proximity.
    3. This avoids selecting a near-date image that is 99% cloud covered.
    """
    target_dt = to_utc_timestamp(target_date)
    window_days = max(SEARCH_WINDOWS_DAYS)
    start_dt = (target_dt - pd.Timedelta(days=window_days)).strftime("%Y-%m-%d")
    end_dt = (target_dt + pd.Timedelta(days=window_days)).strftime("%Y-%m-%d")

    try:
        candidate_items = list(
            client.search(
                collections=[collection],
                bbox=bbox,
                datetime=f"{start_dt}T00:00:00Z/{end_dt}T23:59:59Z",
                max_items=100,
            ).items()
        )
    except Exception as exc:
        raise RuntimeError(f"STAC search failed near {target_date}: {exc}") from exc

    if not candidate_items:
        raise RuntimeError(f"No Sentinel-2 L2A scene found near {target_date}.")

    def score(item):
        acquisition = to_utc_timestamp(item.datetime)
        cloud = item.properties.get("eo:cloud_cover", np.nan)
        return (
            cloud_priority(cloud),
            abs((acquisition - target_dt).total_seconds()),
            np.inf if pd.isna(cloud) else float(cloud),
        )

    return min(candidate_items, key=score)


client = Client.open(STAC_ENDPOINT)

scenes = {}
for act, date in SCENE_DATES.items():
    print(f"Searching {act} ({date})...")
    item = robust_search(client, MATAIAN_BBOX, date)
    scenes[act] = item

    acquisition = to_utc_timestamp(item.datetime).strftime("%Y-%m-%d")
    cloud = item.properties.get("eo:cloud_cover", np.nan)
    cloud_txt = f"{cloud:.1f}%" if pd.notna(cloud) else "N/A"
    print(f"  Selected item: {item.id}")
    print(f"  Acquisition date: {acquisition}")
    print(f"  Reported cloud cover: {cloud_txt}")

pre_item = scenes["Pre"]
mid_item = scenes["Mid"]
post_item = scenes["Post"]

print("✓ Scene search complete")
print(f"  Scenes found: {list(scenes.keys())}")


In [ ]:
# [S3] Load and Composite Multi-Spectral Cubes
# ──────────────────────────────────────────────────────────────────
# Load Sentinel-2 reflectance bands and create stretched RGB composites

# Reference bands: B02 (Blue), B03 (Green), B04 (Red), B08 (NIR), B11 (SWIR)
BANDS = ["B02", "B03", "B04", "B08", "B11"]
BAND_NAMES = {
    "B02": "Blue",
    "B03": "Green",
    "B04": "Red",
    "B08": "NIR",
    "B11": "SWIR",
}


def stream_cube(scene_item, bands, bbox=MATAIAN_BBOX):
    """Load Sentinel-2 bands and return a cube shaped as (H, W, C)."""
    stacked = _stack_scene(scene_item, assets=bands, bbox=bbox)
    cube = np.moveaxis(stacked.values.astype("float32"), 0, -1) / 10000.0
    grid = _extract_grid_meta(stacked)
    return cube, grid


def composite_stretched(cube, r_idx=2, g_idx=1, b_idx=0, q=(2, 98)):
    """Create an RGB composite with percentile-based contrast stretching."""
    rgb = cube[:, :, [r_idx, g_idx, b_idx]].copy()
    for i in range(3):
        band = rgb[:, :, i]
        p_lo, p_hi = np.nanpercentile(band, q)
        if np.isclose(p_hi, p_lo):
            rgb[:, :, i] = 0
        else:
            rgb[:, :, i] = np.clip((band - p_lo) / (p_hi - p_lo), 0, 1)
    return rgb


cubes = {}
composites = {}
clear_masks = {}
scene_data_masks = {}
scene_valid_masks = {}
scene_grids = {}

for act in ["Pre", "Mid", "Post"]:
    print(f"Loading {act} scene...")
    cubes[act], scene_grids[act] = stream_cube(scenes[act], BANDS)
    composites[act] = composite_stretched(cubes[act])

    # Raw valid pixels come only from data availability (no cloud screening yet).
    scene_data_masks[act] = np.all(np.isfinite(cubes[act]), axis=-1)

    # Clear mask is derived strictly from the SCL classes required by the lab.
    clear_masks[act] = stream_scl(scenes[act])
    scene_valid_masks[act] = scene_data_masks[act] & clear_masks[act]

if len({cube.shape[:2] for cube in cubes.values()}) != 1:
    raise ValueError("All scenes must share the same output grid for change detection.")

grid_meta = scene_grids["Pre"]
lake_rows, lake_cols = bbox_to_pixel_window(LAKE_BBOX_LONLAT, grid_meta)
scene_extent = [MATAIAN_BBOX[0], MATAIAN_BBOX[2], MATAIAN_BBOX[1], MATAIAN_BBOX[3]]
lake_extent = [LAKE_BBOX_LONLAT[0], LAKE_BBOX_LONLAT[2], LAKE_BBOX_LONLAT[1], LAKE_BBOX_LONLAT[3]]

# ── Cloud masking (CRITICAL — without this, ΔNDWI shows "phantom water") ──
clear_pre = clear_masks["Pre"]
clear_mid = clear_masks["Mid"]
clear_post = clear_masks["Post"]

# ── Per-scene valid masks (for Three-Act visualization) ──
valid_pre = scene_valid_masks["Pre"]
valid_mid = scene_valid_masks["Mid"]
valid_post = scene_valid_masks["Post"]

# ── Intersection mask (for difference maps where BOTH dates must be valid) ──
valid_raw = scene_data_masks["Pre"] & scene_data_masks["Mid"] & scene_data_masks["Post"]
valid = valid_raw & clear_pre & clear_mid & clear_post

pair_valid_raw = {
    "Mid-Pre": scene_data_masks["Mid"] & scene_data_masks["Pre"],
    "Post-Mid": scene_data_masks["Post"] & scene_data_masks["Mid"],
}
pair_valid = {
    "Mid-Pre": valid_mid & valid_pre,
    "Post-Mid": valid_post & valid_mid,
}

print("✓ Cubes loaded and composited")
print(f"  Cube shape: (H, W, C) = {cubes['Pre'].shape}")
for act in ["Pre", "Mid", "Post"]:
    clear_pct = 100 * scene_valid_masks[act].mean()
    print(f"  {act} clear-sky fraction: {clear_pct:.2f}%")
for label in ["Mid-Pre", "Post-Mid"]:
    overlap_pct = 100 * pair_valid[label].mean()
    print(f"  {label} pairwise valid overlap: {overlap_pct:.2f}%")


### 光譜指標計算：NDVI 與 NDWI

在這一步，我們會從 Sentinel-2 多波段資料中計算兩個最常用的遙測指標，用來描述植生與水體。

---

### 指標公式

**NDVI（Normalized Difference Vegetation Index）**

\[
NDVI = \frac{NIR - Red}{NIR + Red}
\]

**NDWI（Normalized Difference Water Index）**

\[
NDWI = \frac{Green - NIR}{Green + NIR}
\]

---

### 為什麼要同時使用兩個指標？

雖然本次案例重點是堰塞湖與水體判釋，但若只看單一指標，可能會漏掉地表變化的其他訊號。  
因此我們同時使用 NDVI 與 NDWI，從不同光譜角度觀察地表。

- **NDVI**
  - 對植生覆蓋、葉綠素活性與生長狀態敏感
  - 對水體通常呈現低值，但不一定是最清楚的水體指標
  - 在災害事件中，可輔助辨識植生被破壞、裸露化或受淹區域

- **NDWI**
  - 對開放水體、積水區與濕地更敏感
  - 在水體形成或擴張時通常會顯示較高值
  - 本次堰塞湖案例中，往往比 NDVI 更能清楚表現湖區變化

---

### 在本案例中的判讀思路

- 如果某區在 **災中 NDWI 明顯高於災前**，通常代表水體增加
- 如果某區在 **災中 NDVI 下降**，可能代表植生被水覆蓋、崩塌或裸露地增加
- 如果 **災後 NDWI 再次下降**，可能代表湖水消退或排水恢復

---

### 注意事項

- 指標值會受到雲、雲影、地形陰影與混合像元影響
- 湖岸邊界常同時包含水、土與植生，容易出現光譜混合
- 因此必須結合 SCL 遮罩與後續驗證，不能只憑單張指標圖下結論


In [ ]:
# [S4] Compute NDVI and NDWI for All Three Scenes
# ──────────────────────────────────────────────────────────────────


def ndvi(cube):
    """Compute NDVI from cube (B04=Red at idx 2, B08=NIR at idx 3)."""
    red = cube[:, :, 2]
    nir = cube[:, :, 3]
    return (nir - red) / (nir + red + 1e-8)


def ndwi(cube):
    """Compute NDWI from cube (B03=Green at idx 1, B08=NIR at idx 3)."""
    green = cube[:, :, 1]
    nir = cube[:, :, 3]
    return (green - nir) / (green + nir + 1e-8)


indices_raw = {}
indices = {}

for act in ["Pre", "Mid", "Post"]:
    scene_ndvi = ndvi(cubes[act])
    scene_ndwi = ndwi(cubes[act])

    # "Raw" indices keep all finite pixels so we can demonstrate phantom water.
    raw_mask = scene_data_masks[act]
    indices_raw[act] = {
        "NDVI": np.where(raw_mask, scene_ndvi, np.nan),
        "NDWI": np.where(raw_mask, scene_ndwi, np.nan),
    }

    # Analysis-ready indices additionally require SCL clear-sky masking.
    clear_mask = scene_valid_masks[act]
    indices[act] = {
        "NDVI": np.where(clear_mask, scene_ndvi, np.nan),
        "NDWI": np.where(clear_mask, scene_ndwi, np.nan),
    }

print("✓ NDVI and NDWI computed")
for act in ["Pre", "Mid", "Post"]:
    print(f"{act} scene:")
    for idx_name in ["NDVI", "NDWI"]:
        idx = indices[act][idx_name]
        print(
            f"  {idx_name}: μ={np.nanmean(idx):.3f}, "
            f"σ={np.nanstd(idx):.3f}, "
            f"[{np.nanmin(idx):.3f}, {np.nanmax(idx):.3f}]"
        )


In [ ]:
# [S5] Create Difference Maps: ΔIndex and Visualize 2×2 Panel
# ──────────────────────────────────────────────────────────────────

d_ndvi = {}
d_ndwi = {}
d_ndwi_raw = {}

pair_lookup = {
    "Mid-Pre": ("Mid", "Pre"),
    "Post-Mid": ("Post", "Mid"),
}

for label, (later, earlier) in pair_lookup.items():
    d_ndvi[label] = np.where(
        pair_valid[label],
        indices[later]["NDVI"] - indices[earlier]["NDVI"],
        np.nan,
    )
    d_ndwi[label] = np.where(
        pair_valid[label],
        indices[later]["NDWI"] - indices[earlier]["NDWI"],
        np.nan,
    )
    d_ndwi_raw[label] = np.where(
        pair_valid_raw[label],
        indices_raw[later]["NDWI"] - indices_raw[earlier]["NDWI"],
        np.nan,
    )


def robust_symmetric_limit(*arrays, q=99):
    """Pick a robust color scale so subtle but real differences remain visible."""
    finite_abs = []
    for arr in arrays:
        vals = np.abs(arr[np.isfinite(arr)])
        if vals.size:
            finite_abs.append(vals)

    if not finite_abs:
        return 0.05

    finite_abs = np.concatenate(finite_abs)
    return max(0.05, float(np.nanpercentile(finite_abs, q)))


d_ndvi_zoom = {label: arr[lake_rows, lake_cols] for label, arr in d_ndvi.items()}
d_ndwi_zoom = {label: arr[lake_rows, lake_cols] for label, arr in d_ndwi.items()}
lake_extent = [
    LAKE_BBOX_LONLAT[0],
    LAKE_BBOX_LONLAT[2],
    LAKE_BBOX_LONLAT[1],
    LAKE_BBOX_LONLAT[3],
]
lon_ticks = np.linspace(lake_extent[0], lake_extent[1], 6)
lat_ticks = np.linspace(lake_extent[2], lake_extent[3], 5)

vmax_ndvi = robust_symmetric_limit(d_ndvi_zoom["Mid-Pre"], d_ndvi_zoom["Post-Mid"])
vmax_ndwi = robust_symmetric_limit(d_ndwi_zoom["Mid-Pre"], d_ndwi_zoom["Post-Mid"])

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
panels = [
    (d_ndvi_zoom["Mid-Pre"], "ΔNDVI (Mid - Pre)", vmax_ndvi),
    (d_ndwi_zoom["Mid-Pre"], "ΔNDWI (Mid - Pre)", vmax_ndwi),
    (d_ndvi_zoom["Post-Mid"], "ΔNDVI (Post - Mid)", vmax_ndvi),
    (d_ndwi_zoom["Post-Mid"], "ΔNDWI (Post - Mid)", vmax_ndwi),
]

for ax, (panel, title, vmax) in zip(axes.flat, panels):
    valid_pct = 100 * np.isfinite(panel).mean()
    im = ax.imshow(
        panel,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=lake_extent,
        origin="upper",
    )
    ax.set_title(f"{title}\nLake AOI, valid overlap={valid_pct:.1f}%", fontsize=12, fontweight="bold")
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.set_xticks(lon_ticks)
    ax.set_yticks(lat_ticks)
    ax.set_xticklabels([f"{tick:.2f}" for tick in lon_ticks])
    ax.set_yticklabels([f"{tick:.2f}" for tick in lat_ticks])
    plt.colorbar(im, ax=ax, shrink=0.85)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/W9_L1_difference_maps.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Difference maps created")
print("  Output: W9_L1_difference_maps.png")


### 錯誤示範：如果不做雲遮罩，會發生什麼事？

在遙測災害分析中，**雲與雲影是最常見、也最危險的誤判來源之一**。  
若直接以多時期影像相減，卻沒有先過濾雲層與陰影，差異圖上往往會出現大量看似「新增水體」或「地表劇烈變化」的假訊號。

這些假訊號在防災實務上非常危險，因為它們可能導致：

- 誤判堰塞湖範圍
- 高估受災面積
- 造成錯誤的資源調度與警戒判斷

---

### 本段要觀察的內容

在正式分析前，請比較以下三種圖面：

1. **未做雲遮罩的 ΔNDWI (`valid_raw`)**
   - 常會出現大片零碎斑塊
   - 很多其實不是水體，而是雲、雲影或無效像素造成的假變化

2. **做了 SCL 雲遮罩的 ΔNDWI (`valid`)**
   - 僅保留真正可用的晴空像素
   - 更接近真實的水體變化分布

3. **聚焦於湖區 AOI 的放大圖 (`LAKE_BBOX_LONLAT`)**
   - 幫助你從全區背景中聚焦堰塞湖本體
   - 更容易判斷湖面擴張與消退位置

---

### 為什麼 SCL 遮罩是必要條件？

本作業要求只保留 SCL 類別：

`SCL_CLEAR_CLASSES = [2, 4, 5, 6, 7, 11]`

這代表我們只接受相對可信的地表像素，包括：

- 暗像元
- 植生
- 裸地
- 水體
- 未分類但非雲像元
- 雪／冰

而以下常見類別則會被排除：

- 雲影
- 中高雲
- 薄雲
- 飽和或無資料像元

---

### 實務判讀重點

如果你看到「沒有遮罩時變化很多，但遮罩後大幅消失」，那通常不是模型變差，而是代表：

- 原先大量訊號其實是假的
- 雲遮罩成功移除了不可靠像元
- 你的分析結果因此更有防災決策價值


In [ ]:
# [S5B] Compare ΔNDWI With and Without Cloud Masking
# ──────────────────────────────────────────────────────────────────
# This comparison explicitly demonstrates why the SCL clear-sky mask is mandatory.

compare_vmax = robust_symmetric_limit(
    d_ndwi_raw["Mid-Pre"],
    d_ndwi["Mid-Pre"],
    d_ndwi_zoom["Mid-Pre"],
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

raw_valid_pct = 100 * np.isfinite(d_ndwi_raw["Mid-Pre"]).mean()
masked_valid_pct = 100 * np.isfinite(d_ndwi["Mid-Pre"]).mean()
zoom_valid_pct = 100 * np.isfinite(d_ndwi_zoom["Mid-Pre"]).mean()

im0 = axes[0].imshow(
    d_ndwi_raw["Mid-Pre"],
    cmap="RdBu_r",
    vmin=-compare_vmax,
    vmax=compare_vmax,
    extent=scene_extent,
    origin="upper",
)
axes[0].set_title(f"ΔNDWI Mid-Pre\nWithout SCL Mask (valid={raw_valid_pct:.1f}%)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Longitude (°E)")
axes[0].set_ylabel("Latitude (°N)")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(
    d_ndwi["Mid-Pre"],
    cmap="RdBu_r",
    vmin=-compare_vmax,
    vmax=compare_vmax,
    extent=scene_extent,
    origin="upper",
)
axes[1].set_title(f"ΔNDWI Mid-Pre\nWith SCL Mask (valid={masked_valid_pct:.1f}%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Longitude (°E)")
axes[1].set_ylabel("Latitude (°N)")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(
    d_ndwi_zoom["Mid-Pre"],
    cmap="RdBu_r",
    vmin=-compare_vmax,
    vmax=compare_vmax,
    extent=lake_extent,
    origin="upper",
)
axes[2].set_title(f"ΔNDWI Mid-Pre\nLake AOI Zoom (valid={zoom_valid_pct:.1f}%)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Longitude (°E)")
axes[2].set_ylabel("Latitude (°N)")
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/W9_L1_cloud_mask_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Cloud-mask comparison created")
print("  Output: W9_L1_cloud_mask_comparison.png")


### 討論：哪一個指標最能清楚顯示堰塞湖？

在完成 2×2 差異圖後，請不要只看「哪張圖比較漂亮」，而要思考不同指標背後代表的地表過程。  
本題的重點不是比較顏色多寡，而是比較**地物解釋能力**。

---

### 觀察方向

1. **ΔNDVI（Mid - Pre）**

   請觀察堰塞湖形成區在災中相較災前的 NDVI 變化：

   - 若原本有植生，湖水形成後 NDVI 可能下降
   - 但水體不一定會讓 NDVI 變成極端低值，尤其在混合像元區域
   - 因此 NDVI 常能反映「植生受影響」，但不一定能精準描出湖面

2. **ΔNDWI（Mid - Pre）**

   這通常是本案例最關鍵的圖：

   - 若堰塞湖在災中形成，湖區會呈現明顯正值
   - 河道、積水區與湖體擴張位置應比 NDVI 更清楚
   - 若配合雲遮罩，通常能更直接表現水體訊號

3. **恢復階段（Post - Mid）**

   觀察湖水在災後是否消退：

   - 若水體縮小，`ΔNDWI (Post - Mid)` 常會在原湖區顯示負值
   - 若植生逐步恢復，`ΔNDVI (Post - Mid)` 可能轉為正值
   - 但此階段常受裸露地、崩塌地與混合像元影響，判讀需更小心

---

### 建議回答方向

你可以從以下幾個角度來比較 NDVI 與 NDWI：

- 哪一個指標對湖體邊界更敏感？
- 哪一個指標較不容易受到周邊植生干擾？
- 哪一個指標在災中與災後的時序變化比較符合地貌過程？
- 哪一個指標在防災應用中更容易轉換成「可行動」的資訊？

---

### 反思問題

- 哪一個指標對大氣雜訊、岸邊植生與混合像元較穩健？
- 為什麼 NDWI 會比 NDVI 更適合做水體偵測？
- 若你希望「盡量不要漏掉水體」，你會選擇哪個指標做主判斷？
- 若你希望「只保留最可信的湖區」，又會如何調整判讀策略？


In [ ]:
# [S6] Threshold Sensitivity Demo
# ──────────────────────────────────────────────────────────────────
# Sweep NDWI thresholds and measure how much water area is detected.


def detect_water(index_map, threshold, valid_mask=None):
    """Count pixels where NDWI > threshold."""
    water = index_map > threshold
    if valid_mask is not None:
        water = water & valid_mask
    return int(np.count_nonzero(water))


thresholds = np.linspace(0.00, 0.25, 26)
detected_areas = {}

for act in ["Mid", "Post"]:
    areas = []
    for t in thresholds:
        pixel_count = detect_water(indices[act]["NDWI"], t, scene_valid_masks[act])
        areas.append(pixel_count * PIXEL_AREA_KM2)
    detected_areas[act] = np.array(areas)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, detected_areas["Mid"], "o-", label="Mid scene (lake peak)", linewidth=2)
ax.plot(thresholds, detected_areas["Post"], "s-", label="Post scene (after drainage)", linewidth=2)
ax.axvline(DEFAULT_NDWI_THRESHOLD, color="red", linestyle="--", label=f"Reference threshold = {DEFAULT_NDWI_THRESHOLD:.2f}", alpha=0.8)
ax.set_xlabel("NDWI Threshold", fontsize=12)
ax.set_ylabel("Detected Water Area (km²)", fontsize=12)
ax.set_title("Threshold Sensitivity: Detected Water Area vs NDWI Threshold", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/W9_L1_threshold_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Threshold sensitivity analysis complete")
print(f"  Mid-event water area at NDWI>{DEFAULT_NDWI_THRESHOLD:.2f}: {detected_areas['Mid'][np.argmin(np.abs(thresholds - DEFAULT_NDWI_THRESHOLD))]:.3f} km²")
print(f"  Post-event water area at NDWI>{DEFAULT_NDWI_THRESHOLD:.2f}: {detected_areas['Post'][np.argmin(np.abs(thresholds - DEFAULT_NDWI_THRESHOLD))]:.3f} km²")
print("  Output: W9_L1_threshold_sensitivity.png")


### 討論：閾值不是公式答案，而是決策選擇

在遙測分類或變遷偵測中，閾值（threshold）往往被誤解成一個固定且唯一的正確答案。  
但在防災實務裡，閾值其實是**風險偏好與應用目標的體現**。

換句話說，同一張 NDWI 圖，面對不同任務時，最佳閾值可能不同。

---

### 核心觀念

- **低閾值**：較容易把疑似水體都抓出來
  - 優點：敏感、較不容易漏掉災情
  - 缺點：可能誤報較多

- **高閾值**：只保留最明顯的水體像元
  - 優點：較保守、誤報較少
  - 缺點：可能漏掉邊界區、淺水區或混合像元

---

### 不同情境下的閾值策略

| 使用情境 | 建議閾值 | 判斷邏輯 | 優先考量 |
|----------|----------|----------|----------|
| **災害警戒 / 早期預警** | 低（約 0.05） | 寧可多抓一些疑似區，不要漏掉危險區 | 敏感度、低漏判 |
| **保險理賠 / 行政核定** | 中（約 0.10） | 兼顧檢出率與誤判率 | 平衡 |
| **科學製圖 / 歷史建檔** | 高（約 0.15） | 僅保留高可信度水體 | 特異性、低誤判 |

---

### 在本案例中應該怎麼想？

對於馬太鞍堰塞湖事件：

- 若你是**災中應變單位**，通常更在意「有沒有漏掉可能淹水區」
- 若你是**災後調查或學術分析**，可能更在意「地圖是否精準、是否少誤判」

因此，最佳閾值不只是數學問題，而是管理問題。

---

### 建議反思

- 在 2025 年 10 月災後階段，應該採用較低還是較高的 NDWI 閾值？
- 若採用較低閾值，救災單位會多付出哪些成本？
- 若採用較高閾值，又可能漏掉哪些仍有風險的區域？
- 你會如何向防災決策者解釋閾值選擇背後的風險？


## Lab 2：精度評估與驗證

### 實驗目標

在 Lab 1 中，我們已經能從影像中找出疑似水體與變化區，但這還不夠。  
真正可用於災害判斷的結果，必須回答一個更重要的問題：

**「這張圖到底可信嗎？」**

因此，本實驗的重點是把影像判釋結果和地面驗證資料對照，量化模型表現，並將結果轉換成有決策意義的指標。

---

### ARIA v6.0 架構

本實驗以 **ARIA v6.0** 為概念框架：

- **Auditor**：用混淆矩陣檢查預測與真值的對應關係
- **Rater**：以 PA、UA、OA、Kappa 等指標量化模型表現
- **Indicator**：以信心圖呈現不同區域的不確定性
- **Advisor**：將結果整理成可供應變決策參考的報告

---

### 主要工作流程

1. 載入 `validation_points.geojson` 驗證點資料
2. 根據 Lab 1 的 NDWI 閾值建立二值化水體遮罩
3. 將驗證點的經緯度轉換為影像像素位置
4. 在遮罩中抽取每個點的預測值
5. 建立混淆矩陣（Confusion Matrix）
6. 計算精度指標：
   - Producer's Accuracy（PA）
   - User's Accuracy（UA）
   - Overall Accuracy（OA）
   - Cohen's Kappa
   - F1-score
7. 使用 F1-score 掃描最佳閾值
8. 建立三階信心分布圖（High / Low / No Signal）
9. 整理 ARIA v6.0 驗證報告

---

### 本實驗最重要的概念

在災害分析中，**精度高不代表結果有用**。  
若水體只占研究區很小一部分，一個永遠預測「不是水」的模型也可能得到很高的 OA，但它對災害應變毫無幫助。

所以本實驗特別強調：

- 為什麼不能只看 OA
- 為什麼要同時看 PA、UA 與 Kappa
- 為什麼 F1-score 可作為閾值選擇的輔助依據
- 為什麼信心分布圖比單一二值化結果更有決策價值


### 如何建立可信的地面驗證資料？

本次實驗使用的是教師提供的 **60 個校正後驗證點**，這些點位已經過人工檢核，可作為示範用的地面真值資料。  
但在真實的遙測專案中，驗證資料不能只是「順手找幾個點」，而必須是**獨立、可追溯、具有代表性**的資料集。

---

### 本作業使用的驗證點組成

驗證點共 60 筆，包含：

- `lake`：代表水體或湖區，視為 1
- `landslide`：代表崩塌或非水體，視為 0
- `stable`：代表穩定地表，視為 0

這樣的設計可讓我們檢驗模型是否真的能把湖區與非湖區區分開來。

---

### 真實專案中常見的地面真值來源

| 方法 | 說明 | 成本 | 適用情境 |
|------|------|------|----------|
| **現地調查 + GPS** | 到現場記錄點位與實際地表狀態 | 高 | 小範圍、最高可信度 |
| **UAV / 無人機影像** | 取得高解析空拍圖作為對照 | 中 | 中等範圍、災後巡查 |
| **Google Earth Pro 時序影像** | 比對災前後高解析度影像 | 低 | 歷史事件回顧、桌上驗證 |
| **NCDR 災害報告** | 使用國家災害防救科技中心資料 | 低 | 臺灣災害案例 |
| **Copernicus EMS** | 使用國際緊急製圖資料 | 低 | 大型國際災害 |
| **新聞與地理標記照片** | 交叉比對現地報導與照片 | 低 | 快速初步參考 |

---

### 建立驗證資料時的四個原則

1. **獨立性（Independence）**  
   驗證資料應來自與分析影像不同的來源，不能直接拿同一張影像自己驗證自己。

2. **分層抽樣（Stratified Sampling）**  
   點位不能只挑容易判斷的地方，應同時覆蓋：
   - 明確水體
   - 明確非水體
   - 邊界與混合區

3. **足夠樣本數（Sufficient Sample Size）**  
   一般教學示範可用 30–60 點，但若是研究或實務應用，通常建議更多。

4. **時間一致性（Temporal Match）**  
   驗證資料的時間應盡可能接近衛星影像拍攝日期，否則可能不是同一個地表狀態。

---

### 作業延伸建議

若你想讓作業更扎實，可以額外自行使用 Google Earth Pro 或其他高解析影像來源，再補標 20 個以上驗證點，特別是：

- 湖岸邊界
- 崩塌裸露區
- 容易和淺水混淆的地表

這樣做不只會讓精度評估更可信，也能幫助你更深入理解混合像元與分類誤差的來源。


In [ ]:
# [S7] Build Detection Masks (Index-Based Logic)
# ──────────────────────────────────────────────────────────────────
# Threshold NDWI to create binary water masks.

# Define threshold for water detection.
# This is the Lab 1 reference threshold; S12 will later refresh final outputs
# with the F1-optimal threshold from validation.
THRESHOLD_NDWI = DEFAULT_NDWI_THRESHOLD

mask_cmap = ListedColormap(["#f0f0f0", "#2166ac"])
mask_norm = BoundaryNorm([-0.5, 0.5, 1.5], mask_cmap.N)


def build_water_masks(threshold):
    """Build Mid/Post binary masks from NDWI using the requested threshold."""
    return {
        "Mid": ((indices["Mid"]["NDWI"] > threshold) & scene_valid_masks["Mid"]).astype(np.uint8),
        "Post": ((indices["Post"]["NDWI"] > threshold) & scene_valid_masks["Post"]).astype(np.uint8),
    }


def plot_water_masks(mask_dict, threshold, save_path=f"{OUTPUT_DIR}/W9_L2_masks.png"):
    """Plot full-scene and lake-zoom masks, then save the figure."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    im0 = axes[0, 0].imshow(mask_dict["Mid"], cmap=mask_cmap, norm=mask_norm, extent=scene_extent, origin="upper")
    axes[0, 0].set_title(f"Water Mask (Mid, threshold={threshold:.3f})\nFull Scene")
    plt.colorbar(im0, ax=axes[0, 0], ticks=[0, 1], label="Water Class")

    im1 = axes[0, 1].imshow(mask_dict["Post"], cmap=mask_cmap, norm=mask_norm, extent=scene_extent, origin="upper")
    axes[0, 1].set_title(f"Water Mask (Post, threshold={threshold:.3f})\nFull Scene")
    plt.colorbar(im1, ax=axes[0, 1], ticks=[0, 1], label="Water Class")

    im2 = axes[1, 0].imshow(mask_dict["Mid"][lake_rows, lake_cols], cmap=mask_cmap, norm=mask_norm, extent=lake_extent, origin="upper")
    axes[1, 0].set_title("Water Mask (Mid)\nLake AOI Zoom")
    plt.colorbar(im2, ax=axes[1, 0], ticks=[0, 1], label="Water Class")

    im3 = axes[1, 1].imshow(mask_dict["Post"][lake_rows, lake_cols], cmap=mask_cmap, norm=mask_norm, extent=lake_extent, origin="upper")
    axes[1, 1].set_title("Water Mask (Post)\nLake AOI Zoom")
    plt.colorbar(im3, ax=axes[1, 1], ticks=[0, 1], label="Water Class")

    for ax in axes.flat:
        ax.set_xlabel("Longitude (°E)")
        ax.set_ylabel("Latitude (°N)")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


masks = build_water_masks(THRESHOLD_NDWI)
plot_water_masks(masks, THRESHOLD_NDWI)

print(f"✓ Detection masks created (threshold={THRESHOLD_NDWI:.3f})")
print(f"  Mid mask water pixels: {int(masks['Mid'].sum())}")
print(f"  Post mask water pixels: {int(masks['Post'].sum())}")
print(f"  Mid mask water area: {masks['Mid'].sum() * PIXEL_AREA_KM2:.3f} km²")
print(f"  Post mask water area: {masks['Post'].sum() * PIXEL_AREA_KM2:.3f} km²")
print("  Output: W9_L2_masks.png")
print("  Note: final deliverable masks will be refreshed in S12 using the F1-optimal threshold.")


In [ ]:
# [S8] Load Validation Points and Sample Masks
# ──────────────────────────────────────────────────────────────────
#
# The teacher provides validation_points.geojson with 60 points:
#   - 15 lake, 15 landslide, 30 stable
#   - Coordinates corrected by instructor using VHR imagery + NCDR reports
#
# Load the geojson, extract lon/lat from geometry, and prepare sampling helpers.
# ──────────────────────────────────────────────────────────────────

validation_candidates = [
    Path("data/validation_points.geojson"),
    Path("validation_points.geojson"),
]
validation_path = next((path for path in validation_candidates if path.exists()), None)

if validation_path is None:
    raise FileNotFoundError("validation_points.geojson not found in ./data or current directory.")

with open(validation_path, "r", encoding="utf-8") as _f:
    _gj = json.load(_f)

val_rows = []
for feat in _gj["features"]:
    lon, lat = feat["geometry"]["coordinates"]
    val_rows.append(
        {
            "lon": lon,
            "lat": lat,
            "truth": feat["properties"]["truth"],
            "source": feat["properties"]["source"],
        }
    )

validation_points = pd.DataFrame(val_rows)
validation_points["ground_truth"] = validation_points["truth"].map(
    {"lake": 1, "landslide": 0, "stable": 0}
).astype(int)

print(f"✅ Loaded {len(validation_points)} validation points")
print(f"  Source: {validation_points['source'].unique()}")
print("  Ground truth distribution:")
print(validation_points["truth"].value_counts().to_string())


def geo_to_pixel(lon, lat, grid):
    """
    Convert geographic lon/lat (EPSG:4326) into raster row/col indices.

    The imagery is resampled to UTM Zone 51N (EPSG:32651), so we first
    project the point into meters and then convert it to the closest pixel.
    """
    x, y = grid["transformer"].transform(lon, lat)
    col = int(np.round((x - grid["x"][0]) / grid["dx"]))
    row = int(np.round((y - grid["y"][0]) / grid["dy"]))

    if row < 0 or row >= len(grid["y"]) or col < 0 or col >= len(grid["x"]):
        raise ValueError(f"Point ({lon:.5f}, {lat:.5f}) is outside the raster extent.")

    return row, col


def sample_mask_at_points(mask, points_df, grid, pred_col="predicted"):
    """Sample a binary prediction mask at each validation point."""
    sampled = points_df.copy()
    rows = []
    cols = []
    preds = []
    in_bounds = []

    for lon, lat in zip(sampled["lon"], sampled["lat"]):
        try:
            row, col = geo_to_pixel(lon, lat, grid)
            pred = int(mask[row, col])
            inside = True
        except ValueError:
            row, col, pred, inside = np.nan, np.nan, np.nan, False

        rows.append(row)
        cols.append(col)
        preds.append(pred)
        in_bounds.append(inside)

    sampled["row"] = rows
    sampled["col"] = cols
    sampled[pred_col] = preds
    sampled["in_bounds"] = in_bounds
    return sampled


def evaluate_mid_mask(mask_mid, points_df, grid, pred_col="predicted_mid"):
    """Sample the Mid-event mask at validation points and return evaluation arrays."""
    sampled = sample_mask_at_points(mask_mid, points_df, grid, pred_col=pred_col)
    eval_points = sampled[sampled["in_bounds"]].copy()
    ground_truth = eval_points["ground_truth"].astype(int).to_numpy()
    predicted = eval_points[pred_col].astype(int).to_numpy()
    cm = confusion_matrix(ground_truth, predicted, labels=[0, 1])
    return sampled, eval_points, ground_truth, predicted, cm


def compute_metrics_from_cm(cm):
    """Compute required accuracy metrics from a 2x2 confusion matrix."""
    tn, fp, fn, tp = cm.ravel()
    total = tp + tn + fp + fn

    def safe_div(numerator, denominator):
        return float(numerator / denominator) if denominator != 0 else np.nan

    producer_accuracy = safe_div(tp, tp + fn)
    user_accuracy = safe_div(tp, tp + fp)
    overall_accuracy = safe_div(tp + tn, total)
    pe = safe_div(((tp + fn) * (tp + fp) + (tn + fp) * (tn + fn)), total ** 2)
    kappa = safe_div(overall_accuracy - pe, 1 - pe)
    f1_mid = safe_div(2 * user_accuracy * producer_accuracy, user_accuracy + producer_accuracy)

    return {
        "producer_accuracy": producer_accuracy,
        "user_accuracy": user_accuracy,
        "overall_accuracy": overall_accuracy,
        "kappa": kappa,
        "f1": f1_mid,
        "pe": pe,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "total": total,
    }


def plot_confusion_matrix(cm, threshold, save_path=f"{OUTPUT_DIR}/W9_L2_confusion_matrix.png"):
    """Plot and save the confusion matrix heatmap."""
    cm_df = pd.DataFrame(
        cm,
        index=["No Water", "Water"],
        columns=["No Water", "Water"],
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(
        cm_df,
        annot=True,
        fmt="d",
        cmap="Blues",
        linewidths=0.5,
        linecolor="white",
        square=True,
        cbar_kws={"label": "Count"},
        ax=ax,
    )
    ax.set_xlabel("Predicted Class", fontsize=12, fontweight="bold")
    ax.set_ylabel("Ground Truth Class", fontsize=12, fontweight="bold")
    ax.set_title(
        f"Confusion Matrix: Water Detection (NDWI > {threshold:.3f})\nARIA v6.0 Validation",
        fontsize=14,
        fontweight="bold",
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
# [S9] Compute Confusion Matrix
# ──────────────────────────────────────────────────────────────────
# Sample the Mid-event water mask at validation point locations.

validation_points, eval_points, ground_truth, predicted, cm = evaluate_mid_mask(
    masks["Mid"],
    validation_points,
    grid_meta,
    pred_col="predicted_mid",
)

print("✓ Confusion matrix computed from validation points")
print(f"  Points inside raster extent: {len(eval_points)} / {len(validation_points)}")
print(f"  Evaluation threshold: NDWI > {THRESHOLD_NDWI:.3f}")
print("\nConfusion Matrix:")
print(cm)
print("\nLabels: [0=No Water, 1=Water]")
print(f"  TN (True Negatives): {cm[0, 0]}")
print(f"  FP (False Positives): {cm[0, 1]}")
print(f"  FN (False Negatives): {cm[1, 0]}")
print(f"  TP (True Positives): {cm[1, 1]}")


In [ ]:
# [S10] Compute Accuracy Metrics
# ──────────────────────────────────────────────────────────────────
# Calculate Producer's Accuracy, User's Accuracy, Overall Accuracy, and Kappa.

metrics = compute_metrics_from_cm(cm)

# Keep these scalar variables for downstream reporting cells.
tn = metrics["tn"]
fp = metrics["fp"]
fn = metrics["fn"]
tp = metrics["tp"]
producer_accuracy = metrics["producer_accuracy"]
user_accuracy = metrics["user_accuracy"]
overall_accuracy = metrics["overall_accuracy"]
kappa = metrics["kappa"]
f1_mid = metrics["f1"]

print("✓ Accuracy metrics computed\n")
print("=" * 55)
print("ARIA v6.0 VALIDATION METRICS")
print("=" * 55)
print(f"Producer's Accuracy (PA): {producer_accuracy:.3f} ({producer_accuracy * 100:.1f}%)")
print(f"User's Accuracy (UA):     {user_accuracy:.3f} ({user_accuracy * 100:.1f}%)")
print(f"Overall Accuracy (OA):    {overall_accuracy:.3f} ({overall_accuracy * 100:.1f}%)")
print(f"Cohen's Kappa:            {kappa:.3f}")
print(f"F1 Score:                 {f1_mid:.3f}")
print("-" * 55)
print(f"Water prevalence in validation set: {ground_truth.mean() * 100:.1f}%")
print("Note: OA alone can be misleading under class imbalance (accuracy paradox).")
print("=" * 55)


In [ ]:
# [S11] Plot Confusion Matrix Heatmap
# ──────────────────────────────────────────────────────────────────
# Create and save the confusion matrix heatmap.

plot_confusion_matrix(cm, THRESHOLD_NDWI)

print("✓ Confusion matrix heatmap saved")
print("  Output: W9_L2_confusion_matrix.png")


### 討論：這些精度數字對災害應變到底代表什麼？

當我們得到混淆矩陣與各種精度指標後，不能只把它們當成報告上的數字。  
在災害應變中，每一個數值背後都代表著實際的風險與資源配置問題。

以下請從馬太鞍堰塞湖的情境來理解這些指標：

---

### 1. Producer's Accuracy（PA，生產者精度／召回率／敏感度）

這個指標回答的是：

**「所有真實存在的水體中，我們成功找出多少？」**

如果 PA 很高，表示：

- 真實湖區大多有被模型抓到
- 漏判（False Negative）較少
- 對防災來說較安全，因為較不容易忽略實際危險區

如果 PA 很低，表示：

- 模型漏掉不少真正的水體
- 可能低估受災範圍
- 對避難與警戒判斷是很危險的

---

### 2. User's Accuracy（UA，使用者精度／精確率）

這個指標回答的是：

**「所有被我們判成水體的像元中，有多少真的就是水體？」**

如果 UA 很高，表示：

- 模型標出的水體可信度較高
- 誤報（False Positive）較少
- 可減少不必要的調查與錯誤資源投入

如果 UA 很低，表示：

- 模型可能把很多非水體錯判成水體
- 救災單位可能浪費時間去處理其實不危險的區域

---

### 3. Overall Accuracy（OA，整體精度）

OA 看似直觀，但在災害偵測中常常最危險，因為它只是回答：

**「所有預測中，總共有多少比例判對？」**

問題在於：

- 若研究區大部分都是非水體
- 模型即使幾乎都判成非水體
- OA 仍可能看起來很高

因此 OA 必須看，但**不能單獨看**。

---

### 4. Cohen's Kappa

Kappa 的價值在於：

- 它考慮了「隨機猜對」的可能性
- 可以反映模型是否真的超越隨機分類
- 對樣本不平衡時比單純 OA 更有參考價值

若 Kappa 太低，即使 OA 看起來不錯，也代表模型未必真的可靠。

---

### 災害情境下，哪一種錯誤更嚴重？

- **False Negative（漏判）**
  - 真實有水，但模型沒抓到
  - 代表可能漏掉需要警戒、撤離或巡查的危險區

- **False Positive（誤判）**
  - 其實沒水，但模型判成有水
  - 代表可能浪費人力與資源

在多數災害應變情境中，**漏判通常比誤判更危險**。  
因此早期預警系統常會傾向追求較高 PA，即使 UA 稍微下降也可能可以接受。


### 討論：為什麼 Producer's Accuracy 比 Overall Accuracy 更重要？

這一題是本次作業最重要的概念之一，因為它直接連到災害決策中的 **Accuracy Paradox（精度悖論）**。

---

### 假設情境

假設馬太鞍堰塞湖只占研究區的一小部分，例如約 10%，其餘 90% 都是陸域。

如果有一個非常糟糕的模型，永遠都預測：

> 「全部都不是水」

那它的結果可能會是：

- TN = 90%
- FP = 0%
- TP = 0%
- FN = 10%

這時候：

- **OA = 90%**  
  看起來很高，甚至像是一個「不錯的模型」

- **PA = 0%**  
  卻表示它完全沒有抓到任何真正的水體

---

### 這就是 Accuracy Paradox

當類別分布極度不平衡時：

- 一個完全沒抓到災害的模型
- 仍可能得到高 OA

也就是說：

**OA 很高，不代表模型真的有能力辨識災害。**

---

### 為什麼 PA 在防災上更重要？

PA 的核心是：

**「真正的災害區，有沒有被抓到？」**

對防災而言，這比「整體平均判對多少」更重要，因為：

- 救災不能接受把危險區當成安全區
- 避難與巡查最怕漏掉真實受災位置
- 即使多報一些疑似區，通常仍比漏報危險區好

---

### 在本作業中的實務意義

因為堰塞湖只是研究區中的少數地物，所以你在解讀結果時應該：

- **先看 PA**：判斷湖區有沒有被抓到
- **再看 UA**：判斷抓到的水體有多少是可信的
- **最後再看 OA 與 Kappa**：做整體品質檢核

---

### ARIA v6.0 的設計原則

> 寧可多警戒一些安全區，也不要漏掉一個真正有風險的社區。

這句話代表的是災害監測系統的核心倫理：  
在高風險情境下，**敏感度與漏判控制**往往比表面上漂亮的 OA 更重要。


In [ ]:
# [S12] F1 Score vs Threshold: Finding Optimal Detection Threshold
# ──────────────────────────────────────────────────────────────────
# Sweep NDWI thresholds, sample validation points, and find the best F1 score.

thresholds_f1 = np.linspace(0.00, 0.25, 51)
f1_scores = []
producer_curve = []
user_curve = []

rows = eval_points["row"].astype(int).to_numpy()
cols = eval_points["col"].astype(int).to_numpy()

for t in thresholds_f1:
    candidate_mask = build_water_masks(t)["Mid"]
    predicted_t = candidate_mask[rows, cols]

    f1_scores.append(f1_score(ground_truth, predicted_t, zero_division=0))

    cm_t = confusion_matrix(ground_truth, predicted_t, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    producer_curve.append(tp_t / (tp_t + fn_t) if (tp_t + fn_t) else np.nan)
    user_curve.append(tp_t / (tp_t + fp_t) if (tp_t + fp_t) else np.nan)

f1_scores = np.asarray(f1_scores)
producer_curve = np.asarray(producer_curve)
user_curve = np.asarray(user_curve)

optimal_idx = int(np.nanargmax(f1_scores))
optimal_threshold = float(thresholds_f1[optimal_idx])
optimal_f1 = float(f1_scores[optimal_idx])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds_f1, f1_scores, "o-", linewidth=2, markersize=5, label="F1 Score")
ax.axvline(optimal_threshold, color="red", linestyle="--", linewidth=2, label=f"Best threshold = {optimal_threshold:.3f}")
ax.scatter([optimal_threshold], [optimal_f1], color="red", s=180, zorder=5, marker="*")
ax.set_xlabel("NDWI Threshold", fontsize=12)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title("Optimal Threshold Selection via F1 Score", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/W9_L2_f1_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ F1 vs threshold analysis complete")
print(f"  Optimal threshold: {optimal_threshold:.3f}")
print(f"  Optimal F1 score: {optimal_f1:.3f}")
print(f"  PA at optimal threshold: {producer_curve[optimal_idx]:.3f}")
print(f"  UA at optimal threshold: {user_curve[optimal_idx]:.3f}")
print("  Output: W9_L2_f1_threshold.png")

# Refresh final deliverables using the validation-driven optimal threshold.
THRESHOLD_NDWI = optimal_threshold
masks = build_water_masks(THRESHOLD_NDWI)
optimal_mask_mid = masks["Mid"]
plot_water_masks(masks, THRESHOLD_NDWI)

validation_points, eval_points, ground_truth, predicted, cm = evaluate_mid_mask(
    masks["Mid"],
    validation_points,
    grid_meta,
    pred_col="predicted_mid",
)
metrics = compute_metrics_from_cm(cm)

tn = metrics["tn"]
fp = metrics["fp"]
fn = metrics["fn"]
tp = metrics["tp"]
producer_accuracy = metrics["producer_accuracy"]
user_accuracy = metrics["user_accuracy"]
overall_accuracy = metrics["overall_accuracy"]
kappa = metrics["kappa"]
f1_mid = metrics["f1"]

plot_confusion_matrix(cm, THRESHOLD_NDWI)

print("✓ Final masks and validation products refreshed using the optimal threshold")
print(f"  Final deliverable threshold: NDWI > {THRESHOLD_NDWI:.3f}")
print(f"  Refreshed PA / UA / OA / Kappa: {producer_accuracy:.3f} / {user_accuracy:.3f} / {overall_accuracy:.3f} / {kappa:.3f}")


In [ ]:
# [S13] Build 3-Zone Confidence Map: High / Low / None
# ──────────────────────────────────────────────────────────────────
# Create confidence zones using the required NDWI thresholds.

ndwi_mid = indices["Mid"]["NDWI"]
confidence_map = np.zeros_like(ndwi_mid, dtype=np.uint8)

low_confidence = scene_valid_masks["Mid"] & (ndwi_mid > CONFIDENCE_LOW) & (ndwi_mid <= CONFIDENCE_HIGH)
high_confidence = scene_valid_masks["Mid"] & (ndwi_mid > CONFIDENCE_HIGH)

confidence_map[low_confidence] = 1
confidence_map[high_confidence] = 2

confidence_cmap = ListedColormap(["#d9d9d9", "#fdae61", "#2b8cbe"])
confidence_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], confidence_cmap.N)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(confidence_map, cmap=confidence_cmap, norm=confidence_norm)
ax.set_title("ARIA v6.0 Confidence Map: Water Detection Uncertainty", fontsize=14, fontweight="bold")
ax.set_xlabel("X (pixels)")
ax.set_ylabel("Y (pixels)")

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(["No Signal", "Low Confidence", "High Confidence"])
cbar.set_label("Confidence Level")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/W9_L2_confidence_map.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ 3-zone confidence map created")
print(f"  High confidence pixels (NDWI > {CONFIDENCE_HIGH:.2f}): {int((confidence_map == 2).sum())}")
print(f"  Low confidence pixels ({CONFIDENCE_LOW:.2f} < NDWI <= {CONFIDENCE_HIGH:.2f}): {int((confidence_map == 1).sum())}")
print(f"  No signal pixels (NDWI <= {CONFIDENCE_LOW:.2f} or invalid): {int((confidence_map == 0).sum())}")
print("  Output: W9_L2_confidence_map.png")


In [ ]:
# [S14] ARIA v6.0 Validated Disaster Report
# ──────────────────────────────────────────────────────────────────

report = f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║                     ARIA v6.0 VALIDATED DISASTER REPORT                        ║
║              Remote Sensing Analysis & Validation Authority (ARIA)             ║
╚════════════════════════════════════════════════════════════════════════════════╝

CASE STUDY: Matai'an Barrier Lake (Typhoon Colo Response)
Institution: National Taiwan University (NTU)
Analyst: Remote Sensing Lab
Report Date: 2025-10-12

────────────────────────────────────────────────────────────────────────────────

EXECUTIVE SUMMARY
─────────────────
A temporary barrier lake formed in Matai'an valley during Typhoon Colo (Aug 2025).
Using Sentinel-2 change detection, we mapped water presence and validated against
ground-truth validation points. NDWI proved more sensitive than NDVI for water.

STUDY AREA (Bounding Box)
──────────────────────────
West:  121.28° E
East:  121.52° E
South: 23.56° N
North: 23.76° N
Area:  ~360 km²

TEMPORAL ANALYSIS
─────────────────
Pre-event baseline:   2025-06-01 (before typhoon)
Disaster onset:       2025-08-15 (lake at maximum extent)
Post-event recovery:  2025-10-10 (lake drained)

SPECTRAL INDICES USED
─────────────────────
NDVI = (NIR - Red) / (NIR + Red)
  → Sensitivity: Vegetation density, weak water signal
  → ΔNDVIpeak = {np.nanmean(indices["Mid"]["NDVI"]) - np.nanmean(indices["Pre"]["NDVI"]):.4f}

NDWI = (Green - NIR) / (Green + NIR)
  → Sensitivity: Maximum water detection
  → ΔNDWIpeak = {np.nanmean(indices["Mid"]["NDWI"]) - np.nanmean(indices["Pre"]["NDWI"]):.4f}

VALIDATION RESULTS
──────────────────
Validation Points Analyzed:    {len(eval_points)}
Ground Truth Classes:          Water={int(ground_truth.sum())}, No-Water={int(len(ground_truth) - ground_truth.sum())}

Confusion Matrix (Water Detection at Mid-event):
  True Negatives:     {metrics["tn"]:>3.0f}
  False Positives:    {metrics["fp"]:>3.0f}
  False Negatives:    {metrics["fn"]:>3.0f}
  True Positives:     {metrics["tp"]:>3.0f}

ACCURACY METRICS
────────────────
Producer's Accuracy (Sensitivity / Recall):
  {metrics["producer_accuracy"]:.3f} ({metrics["producer_accuracy"] * 100:.1f}%)
  → {int(metrics["fn"])} false negatives (missed water)

User's Accuracy (Precision):
  {metrics["user_accuracy"]:.3f} ({metrics["user_accuracy"] * 100:.1f}%)
  → {int(metrics["fp"])} false positives (false alarms)

Overall Accuracy:
  {metrics["overall_accuracy"]:.3f} ({metrics["overall_accuracy"] * 100:.1f}%)

Cohen's Kappa (Agreement beyond chance):
  {metrics["kappa"]:.3f}
  → Interpretation: Substantial agreement improves trust beyond chance alone

DETECTION THRESHOLD
────────────────────
Selected NDWI Threshold:  {THRESHOLD_NDWI:.3f}
Rationale:               F1-optimized threshold selected from validation sweep
Optimal Threshold (F1):  {optimal_threshold:.3f}
Optimal F1 Score:        {optimal_f1:.3f}

CONFIDENCE LEVELS
──────────────────
High Confidence Zone (NDWI > {CONFIDENCE_HIGH:.2f}):
  → {int((confidence_map == 2).sum())} pixels, highest priority for evacuation planning

Low Confidence Zone ({CONFIDENCE_LOW:.2f} < NDWI ≤ {CONFIDENCE_HIGH:.2f}):
  → {int((confidence_map == 1).sum())} pixels, requires field verification or VHR imagery

No Signal Zone (NDWI ≤ {CONFIDENCE_LOW:.2f}):
  → {int((confidence_map == 0).sum())} pixels, no strong water evidence or masked by invalid data

RECOMMENDATIONS FOR DISASTER RESPONSE
──────────────────────────────────────
1. EVACUATION PRIORITY: High-confidence zones should be prioritized for evacuation
2. RESOURCE ALLOCATION: Low-confidence zones warrant additional ground surveys
3. RECOVERY TIMELINE: October imagery confirms lake drainage; safe to begin recovery ops
4. THRESHOLD TUNING: Producer's Accuracy is critical; prioritize sensitivity over precision

KEY FINDINGS
────────────
✓ NDWI is more reliable than NDVI for barrier lake detection
✓ Sentinel-2 adequate for large-scale (>100 km²) water mapping
✓ Temporal change (Mid - Pre) more robust than absolute thresholds
✓ Ground validation essential for disaster response decisions

LIMITATIONS & CAVEATS
─────────────────────
• Cloud cover reduces valid pixels (typical for tropical typhoon season)
• Shoreline pixels mixed (land-water transitions) introduce uncertainty
• Validation points limited; larger sample recommended for operational use
• Resolution (~10 m) may miss small isolated ponds

────────────────────────────────────────────────────────────────────────────────

ARIA v6.0 ASSESSMENT: ✓ VALIDATED & READY FOR OPERATIONAL USE

Auditor (Confusion Matrix):     ✓ Computed
Rater (Accuracy Metrics):       ✓ Calculated
Indicator (Confidence Zones):   ✓ Mapped
Advisor (Recommendations):      ✓ Provided

Report Status: FINAL
Quality Level: Operational (Ready for Emergency Response)

────────────────────────────────────────────────────────────────────────────────
Generated by ARIA v6.0 at {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

print(report)

# Save report
with open(f"{OUTPUT_DIR}/ARIA_v6_0_Disaster_Report.txt", "w", encoding="utf-8") as f:
    f.write(report)

print("\n✓ Disaster report saved")
print("  Output: ARIA_v6_0_Disaster_Report.txt")


In [ ]:
# [S15] AI Advisor: Prompt Template for Disaster Response Questions# ──────────────────────────────────────────────────────────────────advisor_prompt_template = """ARIA v6.0 AI ADVISOR - Ask Me About Your Disaster Response Questions═══════════════════════════════════════════════════════════════════════You are analyzing Sentinel-2 remote sensing data for Typhoon Colo impact assessment.Use the validation metrics and change maps above to answer operational questions.CONTEXT PROVIDED:  • Confusion matrix with TP/TN/FP/FN  • Producer's & User's Accuracy scores  • 3-zone confidence map (High/Low/No Signal)  • Pre/Mid/Post temporal scenes  • NDVI & NDWI difference mapsSAMPLE QUESTIONS YOU CAN ANSWER:1. "What's the maximum lake extent in the Mid-event scene?"   → Use high-confidence zone from confidence_map; estimate area in km²2. "How accurate is our water detection for evacuation planning?"   → Reference Producer's Accuracy; explain false negatives risk3. "Why might we have false positives along the lakeshore?"   → Vegetation/water mixing; spectral confusion at boundaries4. "Should we trust the Post-event recovery map?"   → Compare Post and Mid NDWI; check confidence zones5. "How would different thresholds affect our evacuation zone?"   → Reference threshold sensitivity plot; trade sensitivity vs precisionYOUR RESPONSE SHOULD:  ✓ Cite specific metrics from the validation report  ✓ Explain the uncertainty and confidence limits  ✓ Recommend actionable next steps  ✗ Avoid over-interpreting noisy pixels  ✗ Don't claim precision beyond 10m resolution────────────────────────────────────────────────────────────────────────────────To use this prompt with an LLM (e.g., ChatGPT):1. Paste this entire block + the ARIA report above2. Add your question about the disaster response3. Ask the AI to respond using only the provided dataExample user query:  "Based on this analysis, which zones should we evacuate first?"Example AI response:  "Based on the high-confidence water detection (NDWI > 0.15), zone XYZ should be   prioritized. However, note that our Producer's Accuracy is 87%, so we may be   missing ~13% of the actual water. Recommend ground surveys in low-confidence zones."────────────────────────────────────────────────────────────────────────────────"""print(advisor_prompt_template)with open(f'{OUTPUT_DIR}/AI_Advisor_Prompt_Template.txt', 'w', encoding='utf-8') as f:    f.write(advisor_prompt_template)print("✓ AI Advisor prompt template saved")print(f"  Output: AI_Advisor_Prompt_Template.txt")

## 實驗完成檢核清單

完成兩個實驗後，請依照下列項目逐一確認。這不只是為了交作業，也是在訓練你檢查遙測結果是否具有可信度與一致性。

---

### Lab 1 應完成項目

- [ ] 已成功計算三期影像的 NDVI 與 NDWI（Pre / Mid / Post）
- [ ] 已產出 2×2 差異圖，包含：
  - `ΔNDVI (Mid - Pre)`
  - `ΔNDWI (Mid - Pre)`
  - `ΔNDVI (Post - Mid)`
  - `ΔNDWI (Post - Mid)`
- [ ] 已完成 NDWI 閾值掃描圖
- [ ] 已能說明為什麼 NDWI 通常比 NDVI 更適合本案例的水體判釋
- [ ] 已確認雲遮罩有正確套用，沒有大面積 phantom water

---

### Lab 2 應完成項目

- [ ] 已建立水體二值化遮罩（Mid / Post）
- [ ] 已將驗證點抽樣到影像像素位置
- [ ] 已建立混淆矩陣
- [ ] 已計算以下精度指標：
  - Producer's Accuracy（PA）
  - User's Accuracy（UA）
  - Overall Accuracy（OA）
  - Kappa
  - F1-score
- [ ] 已完成 confusion matrix 熱力圖
- [ ] 已完成 F1-score vs Threshold 圖
- [ ] 已完成三級信心分布圖（High / Low / No Signal）
- [ ] 已產出 ARIA v6.0 驗證報告

---

### `output/` 資料夾中應該包含的檔案

- [ ] `W9_L1_difference_maps.png`：NDVI / NDWI 差異圖
- [ ] `W9_L1_threshold_sensitivity.png`：閾值敏感度分析圖
- [ ] `W9_L2_masks.png`：二值化水體遮罩圖
- [ ] `W9_L2_confusion_matrix.png`：混淆矩陣熱力圖
- [ ] `W9_L2_f1_threshold.png`：最佳閾值分析圖
- [ ] `W9_L2_confidence_map.png`：三級信心分布圖
- [ ] `ARIA_v6_0_Disaster_Report.txt`：災害驗證報告
- [ ] `AI_Advisor_Prompt_Template.txt`：AI 諮詢提示模板

---

### 課後反思問題

1. 若將 NDWI 閾值從 `0.15` 降到 `0.05`，偵測結果會有什麼改變？
2. 為什麼「時間差異」通常比「單時相絕對值」更適合災害偵測？
3. 若你是防災決策者，你會優先追求敏感度還是精確率？為什麼？
4. 雲與雲影會如何影響混淆矩陣與精度指標？
5. 為什麼地面真值驗證是建立模型可信度的必要條件？

---

## 繳交前請再次確認

> **沒有驗證的地圖，往往比沒有地圖更危險。**

Notebook 可以順利執行，不代表結果一定正確。繳交前請務必檢查：

| 檢查項目 | 你應該問自己的問題 |
|----------|----------------------|
| **精度指標** | OA 是否異常高？Kappa 是否過低？是否可能有精度悖論？ |
| **差異圖** | ΔNDWI 是否真的在湖區顯示變化？還是只是雲與噪訊？ |
| **閾值** | 你能不能解釋為什麼選擇這個閾值？ |
| **混淆矩陣** | TP / FP / TN / FN 是否和圖面觀察一致？ |
| **作業理解** | 你是理解結果後寫下來，還是只是把輸出貼上去？ |

**能跑完，不代表分析正確；能解釋，才代表你真的學會。**

若結果看起來不合理，請回頭檢查：

- 選到的影像日期是否合適
- SCL 雲遮罩是否正確
- 湖區 AOI 是否定位正確
- 驗證點是否有落在影像範圍內
- 閾值是否過高或過低


## 環境設定：`.env` 範例模板

若你需要自訂 STAC 端點、輸出路徑或其他帳號資訊，可以建立 `.env` 檔案來管理環境參數。  
這種做法可以讓 notebook 更乾淨，也比較適合之後移植到其他機器或專案。

---

### `.env` 範例

```bash
# .env template
# 請勿將此檔案提交到 Git 版本控制

# STAC Planetary Computer
STAC_ENDPOINT="https://planetarycomputer.microsoft.com/api/stac/v1"

# Copernicus DataSpace（若之後有需要）
COPERNICUS_USERNAME="your_username"
COPERNICUS_PASSWORD="your_password"

# Optional: Google Earth Engine
GEE_PROJECT="your-gee-project-id"

# Output
OUTPUT_DIR="output"
VALIDATION_DATA="validation_points.geojson"

# Tunable thresholds
THRESHOLD_NDVI=0.2
THRESHOLD_NDWI=0.1
THRESHOLD_NDWI_LOW=0.05
THRESHOLD_NDWI_HIGH=0.15
```

---

### 在 Python 中讀取 `.env`

```python
from dotenv import load_dotenv
import os

load_dotenv(".env")

stac_endpoint = os.getenv("STAC_ENDPOINT")
output_dir = os.getenv("OUTPUT_DIR")
```

---

### 使用 `.env` 的好處

- 可以把可調整參數集中管理
- 不需要每次都去 notebook 手動改字串
- 較適合未來擴充到其他案例研究
- 可避免把帳號密碼直接寫死在程式中

---

### 安全提醒

- 不要把 `.env` 提交到 Git
- 請將 `.env` 加入 `.gitignore`
- 若有 API key 或帳密，請視為敏感資訊妥善管理


## 本次作業結論：學習重點與結果分析

本次作業的核心，不只是完成一份遙測 notebook，而是建立一套從**資料取得、影像前處理、光譜分析、精度驗證，到防災解釋**的完整流程。  
透過馬太鞍堰塞湖案例，我們實際練習了「如何把衛星影像轉化成可支援決策的空間資訊」。

---

### 一、學習重點整理

1. **多時期影像比單時相影像更能描述災害過程**  
   災害不是靜態事件，而是隨時間演變的地表變化。  
   本作業透過 Pre / Mid / Post 三期影像，建立出災前、災中、災後的完整變遷脈絡。

2. **SCL 雲遮罩是變遷偵測的必要條件**  
   若沒有先移除雲與雲影，差異圖很容易出現大量 phantom water，造成錯誤判釋。  
   這一步決定了後續分析是否具備可信度。

3. **NDWI 通常比 NDVI 更適合水體判釋**  
   NDVI 可協助理解植生與裸露地變化，但在堰塞湖這種以水體變化為核心的案例中，NDWI 通常能更直接地突顯湖區擴張與消退。

4. **閾值不是固定答案，而是應用導向的決策**  
   同樣的 NDWI 影像，面對早期預警、災後調查與學術製圖，最佳閾值可能不同。  
   因此必須結合 F1-score、任務需求與風險偏好來選擇。

5. **高 OA 不代表模型真的可靠**  
   在水體只占少數區域時，單看 OA 很容易落入 Accuracy Paradox。  
   真正對防災有用的，是同時理解 PA、UA、Kappa 與 F1 所代表的意義。

6. **信心圖比單一分類圖更有決策價值**  
   三級信心分布圖能將結果分成高可信區、低可信區與無訊號區，更接近真實防災應用中的判斷方式。

---

### 二、結果分析建議

在本作業的正常結果中，通常應觀察到以下幾種現象：

- **災中相對災前（Mid - Pre）**
  - `ΔNDWI` 在堰塞湖形成區應呈現明顯正值
  - `ΔNDVI` 可能下降，反映植生受淹沒或裸露化
  - 湖區邊界與河道附近常出現混合像元，訊號較不穩定

- **災後相對災中（Post - Mid）**
  - `ΔNDWI` 在原湖區可能轉為負值，反映湖水消退
  - `ΔNDVI` 可能出現局部回升，表示地表恢復或植生再生

- **二值化水體遮罩**
  - Mid 時期的水體面積通常應大於 Post 時期
  - 若 Post 仍保有零星水體，可能代表殘餘積水、河道或混合像元

- **精度評估結果**
  - 若 PA 偏高，表示模型較不容易漏掉真實湖區
  - 若 UA 偏低，表示仍有部分非水體被誤判為水體
  - 若 OA 很高但 Kappa 不高，需警覺類別不平衡問題

---

### 三、防災與大地工程角度的解讀

從大地工程防災的角度來看，本次作業並不是單純做圖，而是在回答以下問題：

- 堰塞湖是否真的形成？
- 形成範圍在哪裡？
- 災後是否已經明顯消退？
- 哪些區域可視為高可信風險區？
- 模型結果是否足以支援現地巡查與資源配置？

這些問題都需要同時仰賴：

- 光譜指標判讀
- 雲遮罩品質
- 驗證點精度評估
- 閾值與信心圖判斷

---

### 四、總結

本次作業讓我們從實作中理解到：

> **遙測分析的價值，不在於做出圖，而在於做出可信、可解釋、可行動的圖。**

若能完整完成本次 notebook，你應該已經具備以下能力：

- 能從 STAC 自動取得多時期 Sentinel-2 影像
- 能正確進行 SCL 雲遮罩與水體指標分析
- 能理解並計算 PA、UA、OA、Kappa 與 F1-score
- 能判斷模型是否落入 Accuracy Paradox
- 能把分析結果轉化為信心圖與防災報告

這些能力正是未來進行地形變遷監測、崩塌分析、水體調查與災害風險評估時的重要基礎。
